# Common Hashing Utilities

**Purpose:** Reusable hashing functions for file and data integrity checks.

**Usage:**
```python
%run "./common/common_hashing"

file_hash = calculate_file_hash("/path/to/file.csv")
string_hash = calculate_string_hash("content")
payload_hash = calculate_payload_hash({"key": "value"})
```

**Features:**
* File hashing (SHA256, MD5)
* String hashing
* JSON payload hashing
* DataFrame hashing
* Consistent hash generation across pipeline

In [0]:
import hashlib
import json
from typing import Any, Dict, Optional, Union
from pyspark.sql import DataFrame
from pyspark.sql.functions import md5, sha2, concat_ws, col

# ============================================================================
# HASHING CONFIGURATION
# ============================================================================

DEFAULT_HASH_ALGORITHM = "sha256"  # Options: sha256, md5
MAX_FILE_SIZE_MB = 100  # Maximum file size to hash in memory

In [0]:
def calculate_file_hash(
    file_path: str,
    algorithm: str = DEFAULT_HASH_ALGORITHM,
    max_bytes: Optional[int] = None
) -> str:
    """
    Calculate hash of a file using dbutils.
    
    Args:
        file_path: Full path to file (e.g., /Volumes/...)
        algorithm: Hash algorithm ("sha256" or "md5")
        max_bytes: Maximum bytes to read (None = entire file)
    
    Returns:
        Hex digest of file hash
    
    Example:
        file_hash = calculate_file_hash(
            "/Volumes/workspace/ecommerce/raw_landing/orders.csv"
        )
    
    Note:
        Uses dbutils.fs.head() for reading files.
        For large files, consider using max_bytes parameter.
    """
    try:
        # Default max_bytes to 100MB if not specified
        if max_bytes is None:
            max_bytes = MAX_FILE_SIZE_MB * 1024 * 1024
        
        # Read file content
        file_content = dbutils.fs.head(file_path, maxBytes=max_bytes)
        
        # Calculate hash
        if algorithm == "sha256":
            hash_obj = hashlib.sha256(file_content.encode('utf-8'))
        elif algorithm == "md5":
            hash_obj = hashlib.md5(file_content.encode('utf-8'))
        else:
            raise ValueError(f"Unsupported algorithm: {algorithm}")
        
        return hash_obj.hexdigest()
    
    except Exception as e:
        # Return error marker instead of raising
        # This allows pipeline to continue with warning
        return f"ERROR_HASHING_{algorithm.upper()}"

def calculate_file_hash_spark(
    file_path: str,
    algorithm: str = "sha256"
) -> str:
    """
    Calculate file hash using Spark (for very large files).
    
    Args:
        file_path: Full path to file
        algorithm: Hash algorithm ("sha256" or "md5")
    
    Returns:
        Hex digest of file hash
    
    Example:
        # For files > 100MB
        file_hash = calculate_file_hash_spark(
            "/Volumes/workspace/ecommerce/raw_landing/large_file.csv",
            algorithm="md5"
        )
    """
    try:
        # Read file as binary
        df = spark.read.format("binaryFile").load(file_path)
        
        # Calculate hash based on algorithm
        if algorithm == "sha256":
            hash_col = sha2(col("content"), 256)
        elif algorithm == "md5":
            hash_col = md5(col("content"))
        else:
            raise ValueError(f"Unsupported algorithm: {algorithm}")
        
        # Get hash value
        hash_value = df.select(hash_col.alias("hash")).first()["hash"]
        return hash_value
    
    except Exception as e:
        return f"ERROR_HASHING_{algorithm.upper()}"

In [0]:
def calculate_string_hash(
    content: str,
    algorithm: str = DEFAULT_HASH_ALGORITHM
) -> str:
    """
    Calculate hash of a string.
    
    Args:
        content: String content to hash
        algorithm: Hash algorithm ("sha256" or "md5")
    
    Returns:
        Hex digest of string hash
    
    Example:
        string_hash = calculate_string_hash("hello world")
    """
    if algorithm == "sha256":
        hash_obj = hashlib.sha256(content.encode('utf-8'))
    elif algorithm == "md5":
        hash_obj = hashlib.md5(content.encode('utf-8'))
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")
    
    return hash_obj.hexdigest()

def calculate_payload_hash(
    payload: Union[Dict[str, Any], list],
    algorithm: str = DEFAULT_HASH_ALGORITHM,
    sort_keys: bool = True
) -> str:
    """
    Calculate hash of JSON payload (dict or list).
    
    Args:
        payload: Dictionary or list to hash
        algorithm: Hash algorithm ("sha256" or "md5")
        sort_keys: Sort dictionary keys for consistent hashing
    
    Returns:
        Hex digest of payload hash
    
    Example:
        payload_hash = calculate_payload_hash({
            "id": 123,
            "name": "John",
            "timestamp": "2024-01-01"
        })
    
    Note:
        sort_keys=True ensures consistent hashing regardless of
        dictionary key order.
    """
    # Convert to JSON string with consistent formatting
    json_string = json.dumps(payload, sort_keys=sort_keys, separators=(',', ':'))
    
    return calculate_string_hash(json_string, algorithm)

In [0]:
def add_row_hash_column(
    df: DataFrame,
    columns: Optional[list] = None,
    hash_column_name: str = "row_hash",
    algorithm: str = "sha256"
) -> DataFrame:
    """
    Add a hash column to DataFrame based on row values.
    
    Args:
        df: Input DataFrame
        columns: Columns to include in hash (None = all columns)
        hash_column_name: Name of the new hash column
        algorithm: Hash algorithm ("sha256" or "md5")
    
    Returns:
        DataFrame with added hash column
    
    Example:
        # Hash all columns
        df_with_hash = add_row_hash_column(df)
        
        # Hash specific columns
        df_with_hash = add_row_hash_column(
            df,
            columns=["id", "name", "email"],
            hash_column_name="customer_hash"
        )
    
    Use Cases:
        * Change detection (compare hashes to find modified records)
        * Deduplication (identify duplicate records)
        * Data lineage tracking
    """
    # Use all columns if not specified
    if columns is None:
        columns = df.columns
    
    # Concatenate column values
    concat_col = concat_ws("|", *[col(c).cast("string") for c in columns])
    
    # Apply hash function
    if algorithm == "sha256":
        hash_col = sha2(concat_col, 256)
    elif algorithm == "md5":
        hash_col = md5(concat_col)
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")
    
    return df.withColumn(hash_column_name, hash_col)

In [0]:
def compare_hashes(hash1: str, hash2: str) -> bool:
    """
    Compare two hashes for equality.
    
    Args:
        hash1: First hash
        hash2: Second hash
    
    Returns:
        True if hashes match, False otherwise
    
    Example:
        if compare_hashes(old_hash, new_hash):
            print("Files are identical")
        else:
            print("Files have changed")
    """
    return hash1.lower() == hash2.lower()

def is_valid_hash(
    hash_value: str,
    algorithm: str = DEFAULT_HASH_ALGORITHM
) -> bool:
    """
    Check if a hash string is valid for the given algorithm.
    
    Args:
        hash_value: Hash string to validate
        algorithm: Expected hash algorithm
    
    Returns:
        True if hash is valid format, False otherwise
    
    Example:
        if is_valid_hash(file_hash):
            # Process hash
            pass
    """
    # Check for error markers
    if hash_value.startswith("ERROR_"):
        return False
    
    # Check length based on algorithm
    if algorithm == "sha256":
        expected_length = 64
    elif algorithm == "md5":
        expected_length = 32
    else:
        return False
    
    # Check length and hex format
    if len(hash_value) != expected_length:
        return False
    
    try:
        int(hash_value, 16)  # Verify it's valid hex
        return True
    except ValueError:
        return False